# EXP-007 dev-small 확인: dense 문서 집계 가중치

상위 2개 합(`[1.0, 1.0]`, 처음 승인한 설계)과 max + 0.1·2위(`[1.0, 0.1]`, 바꾼 설계)를 EXP-004와 dev-small에서 비교하고, 손실이 정답 문서의 청크 수와 관계있는지 본다. 판정에 쓰지 않는다. 순위 파일은 `rag.search +exp=exp007 index.scope=dev-small`(합은 `'retriever.doc_agg.weights=[1.0,1.0]' retriever.run_name=hybrid-cond-filter-agg2sum`)로 만든다.

In [1]:
import collections
import json
from pathlib import Path

import pandas as pd

from rag.eval.metrics import gold_ranks, load_cfg, load_gold, read_run

In [2]:
ROOT = Path.cwd().parent
RUNS = ROOT / 'data' / 'runs'
CHUNKS = ROOT / 'data' / 'index' / 'kure-v1-fixed-512-64' / 'dev-small' / 'chunks.jsonl'
RUN_FILES = {
    'EXP-004 (max)': 'kure-v1-fixed-512-64-hybrid-cond-filter-dev-small.csv',
    '[1.0, 1.0] 합': 'kure-v1-fixed-512-64-hybrid-cond-filter-agg2sum-dev-small.csv',
    '[1.0, 0.1]': 'kure-v1-fixed-512-64-hybrid-cond-filter-agg2-dev-small.csv',
}

## 지표

In [3]:
gold = load_gold(load_cfg(), 'dev-small')
ranks = {name: gold_ranks(read_run(RUNS / f), gold, 10) for name, f in RUN_FILES.items()}
print(len(gold), 'queries')
print(pd.DataFrame({name: {f'recall@{k}': (r <= k).mean() for k in (1, 5, 10)}
                    for name, r in ranks.items()}).round(4))

1039 queries
           EXP-004 (max)  [1.0, 1.0] 합  [1.0, 0.1]
recall@1          0.9105        0.6218      0.8845
recall@5          0.9808        0.6978      0.9769
recall@10         0.9875        0.7132      0.9875


## 정답 문서의 청크 수별 Recall@5

In [5]:
with CHUNKS.open(encoding='utf-8') as f:
    n_chunks = collections.Counter(json.loads(line)['doc_id'] for line in f)
size = gold['doc_id'].map(n_chunks).clip(upper=4).rename('정답 청크 수(4+)')
one = sum(v == 1 for v in n_chunks.values()) / len(n_chunks)
print('dev-small 문서 중 청크 1개 비율', round(one, 3))
table = pd.DataFrame({name: (r <= 5).groupby(size).mean() for name, r in ranks.items()})
table.insert(0, 'n', size.value_counts().sort_index())
print(table.round(4))

dev-small 문서 중 청크 1개 비율 0.299
               n  EXP-004 (max)  [1.0, 1.0] 합  [1.0, 0.1]
정답 청크 수(4+)                                              
1            304         0.9704        0.0164      0.9539
2            229         0.9651        0.9520      0.9694
3            212         0.9858        0.9858      0.9858
4            294         1.0000        0.9966      1.0000
